# 04 · `concurrent.futures` in Depth: Executors & Futures

**Module 13 – Concurrency & asyncio** · Estimated time: 45 min

> One API for threads, processes and (3.14+) subinterpreters. This is the recommended high-level way to run work in parallel.

### 🎯 What you will learn
- The `Executor` API: `submit`, `map` (`timeout`, `chunksize`, 🆕 `buffersize` in 3.14), `shutdown`
- `ThreadPoolExecutor`, `ProcessPoolExecutor`, `InterpreterPoolExecutor` options
- `Future` objects: `result`, `exception`, `done`, `cancel`, `add_done_callback`
- Waiting strategies: `as_completed`, `wait(FIRST_COMPLETED | FIRST_EXCEPTION | ALL_COMPLETED)`
- Error handling, timeouts, cancellation and clean shutdown
- Bridging to asyncio with `run_in_executor` / `asyncio.to_thread`

In [1]:
import time, random, threading
from concurrent.futures import (ThreadPoolExecutor, ProcessPoolExecutor, as_completed, wait,
                                FIRST_COMPLETED, FIRST_EXCEPTION, ALL_COMPLETED, TimeoutError as FuturesTimeout)

def fetch(url, delay=None):
    time.sleep(delay if delay is not None else random.uniform(0.05, 0.3))
    if "bad" in url:
        raise ConnectionError(f"cannot reach {url}")
    return f"{url} ok"

URLS = ["a.com", "b.com", "bad.com", "c.com", "d.com"]
random.seed(3)

## 1. `submit()` → `Future`

A **Future** represents a result that will exist later.

In [2]:
with ThreadPoolExecutor(max_workers=3, thread_name_prefix="fetch") as ex:
    fut = ex.submit(fetch, "a.com", 0.2)
    print("running?", fut.running(), "| done?", fut.done())
    print("result:", fut.result(timeout=2))          # blocks until ready (re-raises worker exceptions)
    print("done?", fut.done(), "| exception:", fut.exception())

running? True | done? False


result: a.com ok
done? True | exception: None


| Future method | Meaning |
|---|---|
| `result(timeout=None)` | wait and return the value (or raise the worker's exception / `TimeoutError`) |
| `exception(timeout=None)` | wait and return the exception (or `None`) |
| `done()` / `running()` / `cancelled()` | status |
| `cancel()` | cancel if not started yet; returns `True` on success |
| `add_done_callback(fn)` | call `fn(future)` when it finishes |

## 2. `as_completed`: handle results as soon as each finishes

In [3]:
with ThreadPoolExecutor(max_workers=5) as ex:
    futures = {ex.submit(fetch, url): url for url in URLS}       # map future → input
    for fut in as_completed(futures, timeout=5):
        url = futures[fut]
        try:
            print(f"✅ {fut.result()}")
        except ConnectionError as e:
            print(f"❌ {url}: {e}")

✅ a.com ok
❌ bad.com: cannot reach bad.com
✅ b.com ok


✅ c.com ok
✅ d.com ok


## 3. `map`: ordered results, simple syntax

In [4]:
with ThreadPoolExecutor(max_workers=4) as ex:
    results = ex.map(fetch, ["a.com", "b.com", "c.com"])     # lazy iterator, results in INPUT order
    print(list(results))

try:
    with ThreadPoolExecutor(4) as ex:
        print(list(ex.map(fetch, URLS)))                      # an exception surfaces when its result is reached
except ConnectionError as e:
    print("map raised:", e)

['a.com ok', 'b.com ok', 'c.com ok']


map raised: cannot reach bad.com


🆕 **Python 3.14: `buffersize`.** Plain `map` submits **every** input immediately, which is a problem for huge or infinite iterables. `buffersize=N` keeps at most N tasks in flight:

In [5]:
import itertools
with ThreadPoolExecutor(4) as ex:
    infinite = itertools.count()
    squares = ex.map(lambda x: x * x, infinite, buffersize=8)    # safe even for an infinite input
    print(list(itertools.islice(squares, 10)))

[0, 1, 4, 9, 16, 25, 36, 49, 64, 81]


## 4. `wait`: wait for some or all futures

In [6]:
with ThreadPoolExecutor(5) as ex:
    futures = [ex.submit(fetch, u, d) for u, d in zip(URLS, [0.3, 0.1, 0.2, 0.4, 0.5])]

    done, pending = wait(futures, return_when=FIRST_COMPLETED)
    print("first completed:", [f.result() for f in done], "| still pending:", len(pending))

    done, pending = wait(futures, return_when=FIRST_EXCEPTION)
    print("stopped at first exception:", [str(f.exception()) for f in done if f.exception()])

    done, pending = wait(futures, timeout=10, return_when=ALL_COMPLETED)
    print("all done:", len(done), "pending:", len(pending))

first completed: ['b.com ok'] | still pending: 4


stopped at first exception: ['cannot reach bad.com']


all done: 5 pending: 0


### "First result wins" (hedged requests)

In [7]:
mirrors = {"mirror-eu": 0.3, "mirror-us": 0.1, "mirror-in": 0.2}
with ThreadPoolExecutor() as ex:
    futs = [ex.submit(fetch, name, delay) for name, delay in mirrors.items()]
    done, pending = wait(futs, return_when=FIRST_COMPLETED)
    print("fastest:", next(iter(done)).result())
    for f in pending:
        f.cancel()                # best effort: already-running tasks can't be cancelled

fastest: mirror-us ok


## 5. Timeouts and cancellation

In [8]:
ex = ThreadPoolExecutor(max_workers=1)
slow = ex.submit(fetch, "slow.com", 1.0)
queued = ex.submit(fetch, "queued.com", 0.1)           # waits for the only worker
try:
    slow.result(timeout=0.2)
except FuturesTimeout:
    print("timed out waiting (the task itself keeps running!)")
print("cancel queued task:", queued.cancel(), "| cancel running task:", slow.cancel())
ex.shutdown(wait=True, cancel_futures=True)            # cancel_futures: drop everything not started
print("slow finished anyway:", slow.result())

timed out waiting (the task itself keeps running!)
cancel queued task: True | cancel running task: False


slow finished anyway: slow.com ok


> ⚠️ Threads can't be killed from outside. A timeout only stops **waiting**. Design long tasks to check a stop flag (`threading.Event`) or use processes when you must kill work.

## 6. Callbacks

In [9]:
lock = threading.Lock()
progress = {"done": 0}

def on_done(fut):
    with lock:
        progress["done"] += 1
        status = "ok" if fut.exception() is None else f"error: {fut.exception()}"
        print(f"[callback] {progress['done']}/{len(URLS)} {status}")

with ThreadPoolExecutor(3) as ex:
    for url in URLS:
        ex.submit(fetch, url).add_done_callback(on_done)

[callback] 1/5 error: cannot reach bad.com
[callback] 2/5 ok


[callback] 3/5 ok


[callback] 4/5 ok


[callback] 5/5 ok


Callbacks run in the worker thread (or the calling thread if the future is already done), so keep them short and thread-safe.

## 7. Executor options

| Option | Thread | Process | Interpreter (3.14) |
|---|---|---|---|
| `max_workers` | default `min(32, cpus + 4)` | default `cpus` | default `cpus` |
| `thread_name_prefix` | ✅ | – | ✅ |
| `initializer`, `initargs` | ✅ | ✅ | ✅ |
| `mp_context` | – | ✅ choose `spawn`/`forkserver` | – |
| `max_tasks_per_child` | – | ✅ recycle workers | – |
| best for | I/O-bound | CPU-bound | CPU-bound, lower overhead |

In [10]:
%%writefile cf_tasks.py
import os
def cpu_heavy(n):
    return sum(i * i for i in range(n)), os.getpid()

Writing cf_tasks.py


In [11]:
import os
from cf_tasks import cpu_heavy
with ProcessPoolExecutor(max_workers=4) as ex:
    outputs = list(ex.map(cpu_heavy, [2_000_000] * 4, chunksize=1))
print("results equal:", len({r for r, _ in outputs}) == 1, "| worker pids:", sorted({pid for _, pid in outputs}))

results equal: True | worker pids: [6321, 6322, 6323, 6324]


## 8. Bridging to asyncio

From async code, run blocking functions in a pool without freezing the event loop (lesson 06):

In [12]:
import asyncio

async def main():
    loop = asyncio.get_running_loop()
    with ThreadPoolExecutor(3) as pool:
        results = await asyncio.gather(*(loop.run_in_executor(pool, fetch, u, 0.2) for u in ["x.com", "y.com"]))
    other = await asyncio.to_thread(fetch, "z.com", 0.1)        # shortcut using the default pool
    return results + [other]

print(await main())

['x.com ok', 'y.com ok', 'z.com ok']


## 9. Patterns & tips
- Use `with Executor(...)`: it waits for tasks and shuts the pool down
- Keep a dict `future → input` to know which input failed
- **Bound** the number of in-flight tasks for huge inputs (`buffersize`, or submit in batches)
- Thread pools for I/O (HTTP, DB, files); process/interpreter pools for CPU
- Don't share one pool between unrelated components that could deadlock waiting on each other (a task that submits to the same full pool and waits for the result)

## 📝 Summary
- `submit` → `Future`; `result()` re-raises worker exceptions; `as_completed` streams results; `map` keeps order (🆕 `buffersize`).
- `wait()` with `FIRST_COMPLETED` / `FIRST_EXCEPTION` / `ALL_COMPLETED`; timeouts stop waiting, not the work.
- `shutdown(cancel_futures=True)` drops queued work; callbacks report progress.
- The same API works for threads, processes and subinterpreters, and plugs into asyncio.

## 🏋️ Practice
1. Download 20 fake URLs with a thread pool; print a progress counter via callbacks and a summary of failures.
2. Implement `first_successful(funcs)` that runs functions in parallel and returns the first result that doesn't raise.
3. Process an "infinite" generator of jobs with `map(..., buffersize=10)` and stop after 100 results.